ESKF

In [1]:
%pip install sympy matplotlib scipy astropy sep opencv-python pyproj -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Create an ESKF class for single bearing estimation based in quaternions
# goal is to rotate the body gravity vector into world frame 
import numpy as np 
class ESKF:
    def __init__(self, dt, sigma_acc, sigma_gyro, sigma_acc_bias, sigma_gyro_bias):
        self.dt = dt
        self.g = np.array([0, 0, -9.81])  # Gravity in world frame

        # nominal state initialization
        self.p = np.zeros(3) # position
        self.v = np.zeros(3) # velocity
        self.q = R.from_quat([0,0,0,1]) # quaternion (identity rotation)
        self.ab = np.zeros(3) # accelerometer bias
        self.wb = np.zeros(3) # gyroscope bias

        # error state covariance initialization
        self.P = np.eye(15) * 1e-4

        # continuous noise spectral densities
        self.var_a = sigma_acc ** 2
        self.var_w = sigma_gyro ** 2
        self.var_ab = sigma_acc_bias ** 2
        self.var_wb = sigma_gyro_bias ** 2

    def skew(self, v): # produce skew symmetric matrix
        return np.array([[0, -v[2], v[1]],
                         [v[2], 0, -v[0]],
                         [-v[1], v[0], 0]])

    def predict(self, acc_meas, gyro_meas): 
        dt = self.dt
        R = self.q.as_matrix()

        # no bias measurements
        a_unb = acc_meas - self.ab
        w_unb = gyro_meas - self.wb

        # nominal state prediction
        self.p += self.v * dt + 0.5 * (R @ a_unb) * dt ** 2
        self.v += (R @ a_unb + self.g) * dt

        # quat integration via exponential map
        dq = R.from_rotvec(w_unb * dt)
        self.q = self.q * dq

        # bias estimates remain constant in the prediction step
        #self.ab = self.ab
        #self.wb = self.wb

        # error state system matrix
        F_x = np.eye(15)
        F_x[0:3, 3:6] = np.eye(3) * dt
        F_x[3:6, 6:9] = -R @ self.skew(a_unb) * dt
        F_x[3:6, 9:12] -R * dt # possible error

        F_x[6:9, 6:9] = R.from_rotvec(w_unb * dt).as_matrix().T
        F_x[6:9, 12:15] = -np.eye(3) * dt

        # noise covariance matrix
        Q_diag = np.block([
            self.var_a * dt**2 * np.ones(3), # position noise
            self.var_a * dt**2 * np.ones(3), # velocity noise
            self.var_w * dt**2 * np.ones(3), # rotation noise
            self.var_ab * dt**2 * np.ones(3), # acc bias random walk 
            self.var_wb * dt**2 * np.ones(3) # gyro bias random walk
        ])
        Q = np.diag(Q_diag)

        # covariance propagation
        self.P = F_x @ self.P @ F_x.T + Q

    def pos_update(self, p_meas, R_meas):
        y = p_meas - self.p  # position measurement innovation
        H = np.zeros((3,15))
        H[:, 0:3] = np.eye(3)  # position is directly observed

        S = H @ self.P @ H.T + R_meas  # innovation covariance
        K = self.P @ H.T @ np.linalg.inv(S)  # Kalman gain

        # error state delta x
        delta_x = K @ y  # error state correction

        # update cov with joseph form for numerical stability
        I_KH = np.eye(15) - K @ H  # identity minus Kalman gain times observation matrix
        self.P = I_KH @ self.P @ I_KH.T + K @ R_meas @ K.T  # update covariance

        self.inject_error_state(delta_x)

    def quatnorm(self, q):
        return q / np.linalg.norm(q)  # normalize the quaternion to ensure it remains a valid rotation

    def quatproduct(self, q1, q2): 
        w1, x1, y1, z1 = q1
        w2, x2, y2, z2 = q2
        w = w1*w2 - x1*x2 - y1*y2 - z1*z2
        x = w1*x2 + x1*w2 + y1*z2 - z1*y2
        y = w1*y2 - x1*z2 + y1*w2 + z1*x2
        z = w1*z2 + x1*y2 - y1*x2 + z1*w2
        return np.array([w, x, y, z])

    def inject_error_state(self, delta_x):
        # take out components from delta x
        dp = delta_x[0:3]
        dv = delta_x[3:6]
        dtheta = delta_x[6:9]
        dab = delta_x[9:12]
        dwb = delta_x[12:15]

        #update nominal states
        self.p += dp
        self.v += dv

        # exp map for quaternion composition
        q_err = R.from_rotvec(dtheta)
        self.q = self.q * q_err
        self.q = R.from_quat(self.q.as_quat() / quatnorm(self.q.as_quat()))

        self.ab += dab
        self.wb += dwb

        G = np.eye(15)
        G[6:9, 6:9] = np.eye(3) - 0.5 * self.skew(dtheta)
        self.P = G @ self.P @ G.T

In [ ]:
# call the ESKF class for single bearing estimation based on quaternions
import numpy as np

from ESKF_CNV import ESKF   

# define initial conditions
q0 = np.array([1, 0, 0, 0])  # quat initial condition
R0 = np.array([[1, 0, 0],
                 [0, 1, 0],
                 [0, 0, 1]])  # identity matrix as initial rotation matrix
pos0 = np.array([0, 0, 0])  # initial position
v0 = np.array([0, 0, 0])  # initial velocity
abt0 = np.array([0, 0, 0])  # initial accelerometer bias
wbt0 = np.array([0, 0, 0])  # initial gryo bias
gt0 = np.array([0, 0, 9.81])  # initial gravity vector

# initial KF parameters
P0 = np.eye(4)  # initial state covariance

# assemble initial conditions into the x0 vector:
x0 = np.concatenate([pos0, v0, q0, abt0, wbt0, gt0])



